In [1]:
import black
import jupyter_black

jupyter_black.load(
    lab=False,
    line_length=79,
    target_version=black.TargetVersion.PY39,)

<IPython.core.display.Javascript object>

In [2]:
import numpy as np
from boolean_analysis import calculate_fourier_transform_matrix, parity_of_1s
import pandas as pd
from tqdm import tqdm
import pickle
from boolean_fourier_learner import BooleanFourierLearner
from pathlib import Path
import lzma
import parse
from spin_lattices import KagomeLattice
from utils import make_unpacked_configurations
import matplotlib.pyplot as plt
from heisenberg_hamiltonians import batched_state_info_df, HeisenbergJ1J2
import lattice_symmetries as ls
import seaborn as sns

%matplotlib inline

In [3]:
experiment_dir = Path("kagome24-2022-12-21/")

In [4]:
learners = {}
for file in experiment_dir.glob("fourier-learner-*.pickle.lz"):
    J, i = parse.search("fourier-learner-{}-{:d}.pickle.lz", str(file)).fixed
    J = float(J)
    with lzma.open(file) as f:
        learners[J, i] = pickle.load(f)

learners_ser = (
    pd.DataFrame(dict(learner=learners))
    .reset_index()
    .rename(columns={"level_0": "J", "level_1": "iterations"})
).set_index(["J", "iterations"])["learner"]

for learner in learners.values():
    learner.coeffs_df_ = None

In [5]:
train_sets = {}
for file in experiment_dir.glob("train-*.feather"):
    (J,) = parse.search("train-{}.feather", str(file)).fixed
    J = float(J)
    train_sets[J] = pd.read_feather(file).set_index("index")

In [225]:
J2 = 0.53


lat = KagomeLattice(width=2, height=4)
system = HeisenbergJ1J2(lat, J1=1, J2=J2, use_symmetries=True)
system.get_eigenstates(0)

fourier_basis = ls.SpinBasis(
    system.symmetry_group,
    number_spins=system.number_spins,
    hamming_weight=None,
    spin_inversion=None,
)
fourier_basis.build()
state_info_df = batched_state_info_df(
    fourier_basis, np.arange(2**system.number_spins, dtype="uint64")
)
state_info_df_system = (
    batched_state_info_df(system.basis, system.canonical_basis.states)
    .reset_index()
    .rename(columns={"index": "state"})
)


def mk_test_set(system, train, size=10000):
    state_info_df_system = (
        batched_state_info_df(system.basis, system.canonical_basis.states)
        .reset_index()
        .rename(columns={"index": "state"})
    )
    return np.random.choice(
        state_info_df_system.merge(
            batched_state_info_df(system.basis, np.array(train))
            .reset_index()
            .rename(columns={"index": "train_state"}),
            on="representative",
            how="outer",
        )[lambda x: x["train_state"].isnull()]["state"],
        size=size,
        replace=False,
    )

self.number_spins=24
Symmetry group contains 16 elements
Hilbert space dimension is 85662
Using cached version of eigenvalues / eigenstates from groundstates/HeisenbergJ1J2-KagomeLattice2x4-1.0-0.53-True-1-1.pickle
Ground state energy is -39.2199137373


In [226]:
def sign_overlap(sign_pred, eigenstate):
    amplitudes = np.abs(eigenstate)
    return (np.sign(sign_pred) * amplitudes * eigenstate).sum() / (
        amplitudes**2
    ).sum()

In [227]:
train_sets[J2].shape

(20000, 3)

In [228]:
test_set_size = 10000
train_set_size = 5000
train = train_sets[J2].iloc[:train_set_size]
test_states = mk_test_set(system, train.index, size=test_set_size)
fourier_learner = learners[J2, 200]

In [229]:
keep_coeff = 200
sets = np.array(fourier_learner.get_coeffs_df().iloc[:keep_coeff].index)
# np.random.choice(
#    fourier_learner.get_coeffs_df().index, size=keep_coeff, replace=False
# )
train_x = calculate_fourier_transform_matrix(
    np.array(train.index), sets, number_spins=system.number_spins
)

In [230]:
train_y = np.sign(train["eigenstate_coeff"])

In [231]:
from sklearn.linear_model import LogisticRegression, LinearRegression

In [232]:
model = LinearRegression()
model.fit(train_x, train_y)

LinearRegression()

In [233]:
test_eigenstate = system.get_df_eigenstate(k=0, canonical_basis=True).loc[
    test_states, "eigenstate_coeff"
]

In [234]:
train_eigenstate = system.get_df_eigenstate(k=0, canonical_basis=True).loc[
    train.index, "eigenstate_coeff"
]

In [235]:
test_x = calculate_fourier_transform_matrix(
    np.array(test_states), sets, number_spins=system.number_spins
)

In [236]:
sign_overlap(model.predict(test_x), test_eigenstate)

0.504012685749478

In [237]:
sign_overlap(model.predict(train_x), train_eigenstate)

0.9689640859361587